# Part 1 — The question and the machines

BNJetTag is a binary-weight transformer jet tagger for the CMS Level-1 trigger. This
series documents how the work is done rather than what it found: the machines, the
contracts between them, and the gates a number passes before anyone is allowed to
quote it.

This part covers the research question and the infrastructure. Run the notebook from
the repository root; the code cells read files under `code/`.

## The research question

The CMS Level-1 trigger decides in microseconds, on FPGAs, which proton collisions to
keep. A tagger that runs there competes for two scarce resources: DSP multiplier
blocks, which are few, and LUT logic fabric, which is plentiful.

The hypothesis is that constraining every weight to $\{-1, +1\}$ sidesteps the scarce
one. A multiply-accumulate against a $\pm 1$ weight is not a multiplication; it is an
add or a subtract, and adds map to LUT and carry logic. The binary core should
therefore synthesize to zero DSPs while tagging efficiency stays close to full
precision (`RESEARCH.md` §1).

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
x = rng.normal(size=16).astype(np.float32)                # one activation vector
w = rng.choice([-1.0, 1.0], size=16).astype(np.float32)   # one binary weight column

mac  = float(x @ w)                              # what a DSP would compute
adds = float(x[w > 0].sum() - x[w < 0].sum())    # what LUT and carry fabric computes

print(f"dot product        : {mac:+.6f}")
print(f"adds and subtracts : {adds:+.6f}")
print("identical          :", np.isclose(mac, adds))

### Two experimental axes

- **Weights.** What binarization costs in tagging efficiency, measured against FP32
  and 8-bit (W8A8) baselines trained under the same recipe.
- **Activations.** How far activations can be quantized at fixed binary weights —
  8 bits, then 6, then 4 (the A8 → A6 → A4 ladder) — before efficiency, resources, or
  latency give way.

Everything else in this series is machinery for answering those two questions in a way
that survives hostile reading.

## The method's one commitment

Numbers are guilty until proven innocent. No tagging-efficiency figure is quoted until
it has been recomputed from the stored score arrays; no hardware figure until it has
been re-parsed from the raw synthesis report; no difference between two variants is
called real without an error bar from seed spread and a paired bootstrap.

None of these rules was adopted on principle. Each was installed after an episode in
which its absence produced a wrong number, and part 8 (what broke) keeps the receipts.

## Three machines

The split across machines is epistemic before it is logistical: each machine is the
only place one kind of evidence can be produced, and the laptop — which produces none
of them — is the only place they are held to account.

| Machine | What runs there | What never runs there |
|---|---|---|
| Laptop | Code and config edits, job manifests, preflight, hls4ml export and conversion gates, AUC recomputation from `.npz`, csynth report parsing, figures, documents | Full training; Vitis synthesis |
| NRP Nautilus (Kubernetes, namespace `cms-ml`) | All training jobs; all ROC evaluation jobs | Synthesis; anything that needs persistent storage |
| `mulder` (Vitis HLS 2023.2) | Vitis HLS C-synthesis only | Training; analysis; plotting; any W&B call |

![The three machines and what moves between them. Generated by `tools/make_process_diagrams.py`.](figures/fig_compute_topology.png)

### Four grades of evidence, never conflated

- **Training** yields the validation macro-OvR AUC — a monitor that picks the best
  epoch, never the number of record.
- **Evaluation** yields not a number but score arrays over the held-out split, from
  which the ROC-test AUC (macro one-vs-rest, n = 260,000) is recomputed later.
- **Conversion** yields structural facts, exact where they apply: a weight typed
  `ap_uint<1>` in the emitted headers cannot occupy a DSP.
- **C-synthesis** yields the only numbers that may be called synthesized — estimated
  LUT/FF/DSP/BRAM and latency in cycles, always quoted with their reuse factor (RF).

## One experiment, end to end

An idea is checked against the literature, then frozen into a design: a JSON config
plus a seed, with the arms, the selection rule, and the falsifier registered before
anything launches. Preflight builds every config on CPU so a typo fails in seconds.
Training runs as Kubernetes jobs whose inputs are all pinned. Evaluation scores the
held-out jets into arrays, and the AUC is not believed until an independent pass
recomputes it. Survivors cross into hardware: an export graph with weights pinned to
literal $\pm 1$, two correlation gates, C-synthesis, and a fixed set of report checks.
Only then does a number enter the record with its source path attached.

![One experiment, idea to record. Dashed boxes are the gates a number must pass before it may be quoted anywhere. Generated by `tools/make_process_diagrams.py`.](figures/fig_pipeline_gates.png)

## Where the project stands

At deployable scale the pipeline delivers the thesis measurement: the round-14
eight-constituent binary model synthesizes with literally zero DSPs across the entire
design — every binary matmul, every scale-restoring affine, the attention arithmetic
included — at the maximum-parallelism operating point (RF = 1, fabric-multiplier
variant, `bnjettag/results/r14/hls_r14.md`).

The same report is candid that this operating point's LUT count exceeds the target
device, and that fitting it is an open problem. A measured headline and its measured
caveat in the same breath is the register the whole series tries to hold.

## Getting onto the cluster

Authentication is the kubelogin OIDC flow: the Nautilus kubeconfig is downloaded once
from `nrp.ai/config`, the context is set to `nautilus`, and the first `kubectl` command
opens a CILogon browser login against university credentials. Access tokens expire in
about 30 minutes and refresh automatically, which the older baked-token method did not.

Namespace membership is not self-service; an admin adds each user to `cms-ml`. The
namespace is shared with the whole group, which sets the etiquette: prefix every
resource (everything here is `kai-`), touch only your own pods, and remember that
containers are stateless.

In [ ]:
# runs on: laptop (kubectl against the NRP Nautilus cluster)
!kubectl config use-context nautilus
!kubectl get jobs,pods -n cms-ml -l app=bnjet-r14

### A shared cluster shapes the job spec

- **Right-sized requests, `requests == limits`.** Deployable-scale jobs are data-bound,
  not FLOP-bound: 2 CPU, 8 Gi memory, 16 Gi ephemeral storage (sized to hold the
  ~2.7 GB data tarball transiently). Honest requests schedule faster.
- **A GPU whitelist with tiered preferences**, preferring the abundant mid-tier
  (A10, L4, 2080 Ti, V100, A40, A5000, 3090) and de-preferring A100-class parts; a
  $d_\text{model} = 32$ model gains nothing from a flagship.
- **A hostname denylist** of known-bad nodes, accumulated empirically.
- **Failure-mode headroom**: `backoffLimit: 2`, so a node-level GPU device-plugin
  admission race does not kill the job, and `activeDeadlineSeconds: 21600`, so a wedged
  pod cannot squat on a GPU indefinitely.

In [ ]:
from pathlib import Path

manifest = Path("code/jobs/kai-bn14-l1x3-n8-w1a8-s1.yaml").read_text().splitlines()
start = next(i for i, line in enumerate(manifest) if line.strip() == "resources:")
print("\n".join(l for l in manifest[start:start + 13] if not l.strip().startswith("#")))

## Nothing persists on the cluster

Since round 6 no job mounts persistent storage. A training pod receives its code as a
ConfigMap, fetches the dataset from Zenodo at startup, trains against an `emptyDir`
scratch volume, and pushes durable outputs to W&B before exiting.

The arrangement was forced before it was appreciated: a cluster-wide storage outage on
2026-07-02 took the ceph-backed volumes down mid-campaign, and the PVC-free rebuild
scheduled better as well. It is now preferred, because every run becomes reproducible
from its job manifest alone — there is no mutable volume whose state a result could
silently depend on. The price is a few minutes of download per pod.

## W&B: provenance and durability, never the number of record

Because pods keep nothing, something else must. Every number quoted anywhere is still
recomputed locally from the primary artifacts, never read from a W&B mirror.

| Field | Value |
|---|---|
| Entity | `kayamaguchi-uc-san-diego` |
| Project | one per campaign: `bnjettag-bitnet` (early rounds, frozen), `bnjettag-final` (era 2), `BNJetTagAug` (round 14) |
| Run name | `<config>-s<seed>` |
| Group | the round or stage, e.g. `r14-n8` |
| Job type | `train` / `distill` / `eval` / `hls` / `dataset` |

Round 14 got its own project deliberately, so that 3-feature numbers can never be
conflated with 16-feature ones (`STATUS-2026-08-08.md` §1.5). The layout is emitted by
one module, `bnjettag/code/hgq2/bnhgq2/wandb_util.py`, so it has a single point of
truth.

### Durability is contractual

The outputs that matter are versioned artifacts: `model-<leaf>` (the checkpoint of
record, with its `train_meta.json` and `input_std.json`), `evaluation-*` (ROC arrays
and tables), `synthesis-*` (raw csynth XML plus the convert manifest), and dataset
reference artifacts that record checksums without uploading data.

The upload blocks on the server-side commit and manifest check and raises on failure,
which training treats as fatal: an `emptyDir` pod that cannot prove its checkpoint is
durable has no business exiting successfully. That blocking check replaced a
hand-rolled retry loop after four checkpoints were lost on 2026-07-15. Hosted sweeps
are deliberately unused, because a sweep controller would bypass pre-registration.

In [ ]:
from pathlib import Path

# the env-var contracts, read straight out of the round-14 job manifest
for line in Path("code/jobs/kai-bn14-l1x3-n8-w1a8-s1.yaml").read_text().splitlines():
    s = line.strip()
    if s.startswith(("export WANDB_", "export BNHGQ2_")):
        print(s)

### The one secret

The W&B API key lives in a single local file, permission-restricted and excluded from
version control, and on the cluster as the Kubernetes secret `kai-wandb`, which job
manifests reference by name. After any rotation the cluster secret must be updated to
match, or every job dies at `wandb.init()`. The key never appears in a manifest, a log,
a commit, or a document.

## mulder: the synthesis box

`mulder` is a group server reachable through an SSH-config alias. It carries Vitis HLS
2023.2, and every synthesis targets part `xcvu13p-flga2577-2-e` (a VU13P) at a 2.5 ns
clock. C-synthesis needs no license; place-and-route would, and is out of scope.

One environment rule matters enough to restate: the full `settings64.sh` must be
sourced, not the HLS-only variant, because the full script provides both `vitis_hls`
and `vitis-run`. The wrapper script does this itself.

In [ ]:
# runs on: laptop (ssh) -> the synthesis itself runs on mulder
!scp myproject_r14n8_rf1.tar.gz mulder:~/csynth/
!ssh mulder 'cd ~/csynth && ./mulder_csynth.sh myproject_r14n8_rf1.tar.gz'

### Memory, not cores, is the constraint

Vitis HLS C-synthesis is single-threaded, so mulder's 64 threads sit idle while one
synthesis pegs one core. What the box actually contributes is 125 GB of RAM: large
syntheses allocate tens of gigabytes, and the 6.4M-parameter monolith needs over 100 GB
on its own. Concurrent runs are governed by a guard that waits for at most two Vitis
processes and at least 40 GB free.

Analysis never happens there. Nothing on mulder is backed up, the box is shared, and
the numbers-of-record rule requires parsing the raw `csynth.xml` where the record
lives. The standing instruction is to bring the raw XML and parsed JSON home as soon as
a synthesis lands, and to log provenance to W&B afterwards, from the laptop.

## Local environments

The laptop runs two virtualenvs. The conversion and evaluation environment is an
exactly pinned mirror of the cluster pods; the plotting environment is separate,
holding the matplotlib toolchain that has no reason to share dependencies with a QAT
export pipeline.

The pins are load-bearing rather than conservative: `da4ml`, needed only for
distributed-arithmetic emissions, drags NumPy below 2.5 through a numba constraint, so
installing it into the main environment would silently change the numeric stack that
produced every verified number.

In [ ]:
import re
from pathlib import Path

txt = Path("code/jobs/kai-bn14-l1x3-n8-w1a8-s1.yaml").read_text()
print("\n".join(re.findall(r'"([A-Za-z0-9_.\[\]-]+==[0-9][0-9A-Za-z.]*)"', txt)))

### Environment-variable contracts

Reproducibility across machines rests on a few environment variables rather than
hardcoded paths, so identical code runs against `/work/data` in a pod and a local
directory on a workstation.

| Variable | Effect |
|---|---|
| `BNHGQ2_TRAIN_DATA` | overrides the config's data path (`bnjettag/code/hgq2/bnhgq2/train.py`) |
| `BNHGQ2_OUT_ROOT` | sets the output root, defaulting to the model store (`bnjettag/code/hgq2/run_stage.py`) |
| `WANDB_PROJECT`, `WANDB_ENTITY`, `WANDB_GROUP`, `WANDB_TAGS` | resolve environment over config over defaults; absent variables reproduce the pre-2026-08 behavior exactly, which is why old manifests stayed valid through the tracking overhaul |

A run is therefore fully determined by its config JSON plus a handful of environment
lines visible in its job manifest.

## Next

Part 2 takes the dataset and its contracts: what the files contain, which jets may ever
touch a reported number, how constituents are ordered before the model sees them, and
which preprocessing must travel with a checkpoint.